# Import

In [1]:
!pip install optuna

In [2]:
!pip install scikit-learn==1.1.3

  Using cached scikit_learn-1.1.3-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (10 kB)
Using cached scikit_learn-1.1.3-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (32.0 MB)
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.7.0
    Uninstalling scikit-learn-1.7.0:
      Successfully uninstalled scikit-learn-1.7.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
scikeras 0.13.0 requires scikit-learn>=1.4.2, but you have scikit-learn 1.1.3 which is incompatible.
imbalanced-learn 0.13.0 requires scikit-learn<2,>=1.3.2, but you have scikit-learn 1.1.3 which is incompatible.
mlxtend 0.23.4 requires scikit-learn>=1.3.1, but you have scikit-learn 1.1.3 which is incompatible.
sklearn-compat 0.1.3 requires scikit-learn<1.7,>=1.2, but you have scikit-learn 1.1.3 which is incompatible.


In [3]:
!pip install scikeras

  Using cached scikit_learn-1.7.0-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (17 kB)
Using cached scikit_learn-1.7.0-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (12.9 MB)
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.1.3
    Uninstalling scikit-learn-1.1.3:
      Successfully uninstalled scikit-learn-1.1.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sklearn-compat 0.1.3 requires scikit-learn<1.7,>=1.2, but you have scikit-learn 1.7.0 which is incompatible.


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
import numpy as np
import tensorflow as tf
import pandas as pd
from scikeras.wrappers import KerasRegressor
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, GRU, Dropout, Input, LayerNormalization, Bidirectional
from tensorflow.keras.losses import mse
from tensorflow.keras.metrics import RootMeanSquaredError, MeanSquaredError
from tensorflow.keras.optimizers import Adam
import matplotlib.pyplot as plt
from keras.callbacks import EarlyStopping
from sklearn.metrics import make_scorer, mean_squared_error
from sklearn.preprocessing import OneHotEncoder, StandardScaler, MinMaxScaler

# 원본 데이터 전처리

In [6]:
# 3개년 데이터 로드 (인덱스 자동추가 제외)
origin_2021 = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/train_subway21.csv', index_col=0)
origin_2022 = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/train_subway22.csv', index_col=0)
origin_2023 = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/train_subway23.csv', index_col=0)

In [7]:
# 칼럼명 변경
origin_2021 = origin_2021.rename(columns={
    'train_subway21.tm': 'time',
    'train_subway21.line': 'line',
    'train_subway21.station_number': 'station_number',
    'train_subway21.station_name': 'station_name',
    'train_subway21.direction': 'direction',
    'train_subway21.stn': 'AWS_num',
    'train_subway21.ta': 'temp',
    'train_subway21.wd': 'wind_direction',
    'train_subway21.ws': 'wind_speed',
    'train_subway21.rn_day': 'rn_day',
    'train_subway21.rn_hr1': 'rn_hr',
    'train_subway21.hm': 'humidity',
    'train_subway21.si': 'solar',
    'train_subway21.ta_chi': 'feel_temp',
    'train_subway21.congestion': 'congestion'
})

origin_2022 = origin_2022.rename(columns={
    'train_subway22.tm': 'time',
    'train_subway22.line': 'line',
    'train_subway22.station_number': 'station_number',
    'train_subway22.station_name': 'station_name',
    'train_subway22.direction': 'direction',
    'train_subway22.stn': 'AWS_num',
    'train_subway22.ta': 'temp',
    'train_subway22.wd': 'wind_direction',
    'train_subway22.ws': 'wind_speed',
    'train_subway22.rn_day': 'rn_day',
    'train_subway22.rn_hr1': 'rn_hr',
    'train_subway22.hm': 'humidity',
    'train_subway22.si': 'solar',
    'train_subway22.ta_chi': 'feel_temp',
    'train_subway22.congestion': 'congestion'
})

origin_2023 = origin_2023.rename(columns={
    'train_subway23.tm': 'time',
    'train_subway23.line': 'line',
    'train_subway23.station_number': 'station_number',
    'train_subway23.station_name': 'station_name',
    'train_subway23.direction': 'direction',
    'train_subway23.stn': 'AWS_num',
    'train_subway23.ta': 'temp',
    'train_subway23.wd': 'wind_direction',
    'train_subway23.ws': 'wind_speed',
    'train_subway23.rn_day': 'rn_day',
    'train_subway23.rn_hr1': 'rn_hr',
    'train_subway23.hm': 'humidity',
    'train_subway23.si': 'solar',
    'train_subway23.ta_chi': 'feel_temp',
    'train_subway23.congestion': 'congestion'
})

In [8]:
# origin dataframe 통합
origin_all = pd.concat([origin_2021, origin_2022, origin_2023], ignore_index=True)
origin_all

,time,line,station_number,station_name,direction,AWS_num,temp,wind_direction,wind_speed,rn_day,rn_hr,humidity,solar,feel_temp,congestion
0,2021010100,1,150,서울역,상선,419,-9.6,291.1,3.3,0.0,0.0,-99.0,-99.0,-12.6,0
1,2021010101,1,150,서울역,상선,419,-9.7,284.6,2.0,0.0,0.0,-99.0,-99.0,-9.8,0
2,2021010105,1,150,서울역,상선,419,-9.3,124.7,2.4,0.0,0.0,-99.0,-99.0,-10.3,1
3,2021010106,1,150,서울역,상선,419,-9.3,126.2,1.7,0.0,0.0,-99.0,-99.0,-10.1,2
4,2021010107,1,150,서울역,상선,419,-9.1,145.7,1.3,0.0,0.0,-99.0,-99.0,-9.7,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16369319,2023123119,8,2828,남위례,하선,572,0.6,0.0,0.0,7.0,0.0,83.1,-99.0,0.0,18
16369320,2023123120,8,2828,남위례,하선,572,0.0,354.7,0.0,7.0,0.0,84.7,-99.0,-0.6,17
16369321,2023123121,8,2828,남위례,하선,572,-0.6,0.0,0.0,7.0,0.0,85.1,-99.0,-1.1,21
16369322,2023123122,8,2828,남위례,하선,572,-0.8,0.0,0.0,7.0,0.0,85.6,-99.0,-1.3,18


In [9]:
# 결측값 → NaN
sentinels = [-99, -99.9, -999.0]
origin_all = origin_all.replace(sentinels, np.nan)

In [10]:
#이상치 처
def find_iqr_outliers(series):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    return (series < lower) | (series > upper)

num_cols = ['temp', 'wind_direction', 'wind_speed', 'rn_day', 'rn_hr', 'humidity', 'solar', 'feel_temp']

for col in num_cols:
    outlier_mask = find_iqr_outliers(origin_all[col])
    origin_all.loc[outlier_mask, col] = np.nan  # 이상치만 NaN 처리

In [11]:
# 결측치 확인
print('결측치 처리 전 결측치 개수')
print(origin_all.isnull().sum())

결측치 처리 전 결측치 개수
time                    0
line                    0
station_number          0
station_name            0
direction               0
AWS_num                 0
temp               216672
wind_direction     230786
wind_speed         714650
rn_day            3241523
rn_hr             1293576
humidity           844594
solar             6064242
feel_temp            1076
congestion              0
dtype: int64


In [12]:
df=origin_all.copy()

In [13]:
df['time'] = pd.to_datetime(df['time'], format='%Y%m%d%H')
df['month'] = df['time'].dt.month
df['hour'] = df['time'].dt.hour

# 2. 겨울/여름 마스크 (벡터화)
winter_mask = df['month'].isin([12, 1, 2])
summer_mask = df['month'].isin([6, 7, 8])
remain = df['month'].isin([3,4,5, 9, 10, 11])
# 3. 밤시간 (벡터화)
df['is_night'] = (
    (winter_mask & ((df['hour'] >= 20) | (df['hour'] < 5))) |
    (summer_mask & ((df['hour'] >= 22) | (df['hour'] < 5))) |
    (remain & ((df['hour'] >= 21) | (df['hour'] < 5)))
)
df.loc[df['is_night'] & df['solar'].isnull(), 'solar'] = int(0)

# 임시 컬럼 제거
df.drop(['month', 'hour', 'is_night'], axis=1, inplace=True)

In [14]:
df = df.sort_values(['AWS_num', 'time']).reset_index(drop=True)

In [15]:
target_cols = ['temp', 'wind_direction', 'wind_speed', 'rn_day', 'rn_hr', 'humidity', 'solar', 'feel_temp']
specific_cols = ['wind_direction', 'rn_day', 'rn_hr']
linear_cols = ['temp', 'wind_speed','humidity', 'solar', 'feel_temp']
# 0단계: 동일 관측소그룹별 specific_cols을 시간순으로 bfill/ffill
print('0단계 시작')
for col in specific_cols:
    df[col] = (
        df.groupby('AWS_num')[col]
        .apply(lambda x: x.bfill().ffill())  # 각 그룹에서만 보간
        .reset_index(drop=True)
    )
print(df.isnull().sum())
print('----------------------------------')

#1단계 :동일 관측소그룹별 linear_cols을 시간순으로 선형보간
print('1단계 시작')
for col in linear_cols:
    df[col] = (
        df
        .sort_values(['AWS_num', 'time'])  # 그룹 내 정렬!
        .groupby('AWS_num')[col]
        .apply(lambda x: x.interpolate(method='linear'))  # 각 그룹에서만 보간
        .reset_index(drop=True)
    )
print(df.isnull().sum())
print('----------------------------------')

print('2단계 시작')
for col in linear_cols:
    # 1) 그룹 내 중앙값으로 채우기
    df[col] = (
        df.groupby('AWS_num')[col]
        .apply(lambda x: x.fillna(x.median()))
        .reset_index(drop=True)
    )
    # 2) 그래도 남으면 전체 중앙값
    df[col] = df[col].fillna(df[col].median())
print(df.isnull().sum())
print('----------------------------------')

0단계 시작
time                    0
line                    0
station_number          0
station_name            0
direction               0
AWS_num                 0
temp               216672
wind_direction          0
wind_speed         714650
rn_day                  0
rn_hr                   0
humidity           844594
solar             2220357
feel_temp            1076
congestion              0
dtype: int64
----------------------------------
1단계 시작
time                   0
line                   0
station_number         0
station_name           0
direction              0
AWS_num                0
temp                4634
wind_direction         0
wind_speed          4636
rn_day                 0
rn_hr                  0
humidity          633392
solar                  0
feel_temp              0
congestion             0
dtype: int64
----------------------------------
2단계 시작
time              0
line              0
station_number    0
station_name      0
direction         0
AWS_num           

In [16]:
# 상/하행 레이블 정의
direction_map = {'상선': 0, '하선': 1, '내선': 2, '외선':3}

# 체감온도 구간 레이블 정의
bins = [-float('inf'), -10, 5, 20, 28, float('inf')]
labels = ["0", "1", "2", "3", "4"]

# 임시 DataFrame temp_df 정의
temp_df = df.copy()

# direction을 숫자 레이블로 변환
temp_df['direction'] = temp_df['direction'].map(direction_map)

# 체감온도 단계 파생변수 추가
temp_df['feel_stage'] = pd.cut(temp_df['feel_temp'], bins=bins, labels=labels).astype(int)

In [17]:
holidays = [
    # 2021
    '20210101','20210211','20210212','20210213','20210301',
    '20210505','20210519','20210606','20210815','20210920',
    '20210921','20210922','20211003','20211009','20211225',
    # 2022
    '20220101','20220131','20220201','20220202','20220301',
    '20220309','20220505','20220508','20220601','20220606',
    '20220815','20220909','20220910','20220911','20220912',
    '20221003','20221009','20221010','20221225',
    # 2023
    '20230101','20230121','20230122','20230123','20230124',
    '20230301','20230505','20230527','20230529','20230606',
    '20230815','20230928','20230929','20230930','20231002',
    '20231003','20231009','20231225'
]
# 공휴일 리스트를 set으로 변환하여 조회 속도 향상
holidays_set = set(holidays)

# 2. 'is_holiday' 컬럼 초기화 (기본값 0: 평일)
temp_df['is_holiday'] = 0

# 3. 주말(토/일) 확인하여 1로 설정
# .dt.dayofweek는 요일을 숫자로 반환합니다 (월=0, 화=1, ..., 토=5, 일=6)
is_weekend = (temp_df['time'].dt.dayofweek >= 5)
temp_df.loc[is_weekend, 'is_holiday'] = 1

# 4. 공휴일 확인하여 1로 설정 (주말과 겹치더라도 1로 유지됨)
# .dt.strftime('%Y%m%d')로 날짜 부분만 YYYYMMDD 문자열로 변환하여 공휴일 리스트와 비교
is_in_holidays = temp_df['time'].dt.strftime('%Y%m%d').isin(holidays_set)
temp_df.loc[is_in_holidays, 'is_holiday'] = 1

In [18]:
temp_df = temp_df.sort_values(['station_number', 'AWS_num','direction','time']).reset_index(drop=True)
temp_df

,time,line,station_number,station_name,direction,AWS_num,temp,wind_direction,wind_speed,rn_day,rn_hr,humidity,solar,feel_temp,congestion,feel_stage,is_holiday
0,2021-01-01 00:00:00,1,150,서울역,0,419,-9.6,291.1,3.3,0.0,0.0,64.1,0.000000,-12.6,0,0,1
1,2021-01-01 01:00:00,1,150,서울역,0,419,-9.7,284.6,2.0,0.0,0.0,64.1,0.000000,-9.8,0,1,1
2,2021-01-01 05:00:00,1,150,서울역,0,419,-9.3,124.7,2.4,0.0,0.0,64.1,0.000000,-10.3,1,0,1
3,2021-01-01 06:00:00,1,150,서울역,0,419,-9.3,126.2,1.7,0.0,0.0,64.1,0.000000,-10.1,2,0,1
4,2021-01-01 07:00:00,1,150,서울역,0,419,-9.1,145.7,1.3,0.0,0.0,64.1,0.000000,-9.7,3,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16369319,2023-12-31 19:00:00,6,9006,응암S,1,400,1.4,0.0,0.0,0.0,0.0,92.1,0.007959,1.3,15,1,1
16369320,2023-12-31 20:00:00,6,9006,응암S,1,400,0.8,308.8,0.5,0.0,0.0,95.3,0.000000,0.7,13,1,1
16369321,2023-12-31 21:00:00,6,9006,응암S,1,400,0.4,320.6,0.1,0.0,0.0,96.2,0.000000,0.2,14,1,1
16369322,2023-12-31 22:00:00,6,9006,응암S,1,400,0.0,117.6,0.1,0.0,0.0,97.1,0.000000,0.0,14,1,1


In [19]:
# 필요한 칼럼만 추출
selected_cols = [
    'time',
    'station_number',
    'direction',
    'temp',
    'wind_speed',
    'rn_hr',
    'humidity',
    'solar',
    'congestion',
    'feel_temp',
    'feel_stage',
    'is_holiday'
]

# temp_df에서 필요한 컬럼만 train_data DataFrame에 저장
train_data = temp_df[selected_cols].copy()

# station_number, direction, time 순으로 정렬
train_data = train_data.sort_values(
    by=['station_number', 'direction', 'time']
).reset_index(drop=True)

# time 제거
train_data = train_data.iloc[:, 1:]

# 완성
train_data

,station_number,direction,temp,wind_speed,rn_hr,humidity,solar,congestion,feel_temp,feel_stage,is_holiday
0,150,0,-9.6,3.3,0.0,64.1,0.000000,0,-12.6,0,1
1,150,0,-9.7,2.0,0.0,64.1,0.000000,0,-9.8,1,1
2,150,0,-9.3,2.4,0.0,64.1,0.000000,1,-10.3,0,1
3,150,0,-9.3,1.7,0.0,64.1,0.000000,2,-10.1,0,1
4,150,0,-9.1,1.3,0.0,64.1,0.000000,3,-9.7,1,1
...,...,...,...,...,...,...,...,...,...,...,...
16369319,9006,1,1.4,0.0,0.0,92.1,0.007959,15,1.3,1,1
16369320,9006,1,0.8,0.5,0.0,95.3,0.000000,13,0.7,1,1
16369321,9006,1,0.4,0.1,0.0,96.2,0.000000,14,0.2,1,1
16369322,9006,1,0.0,0.1,0.0,97.1,0.000000,14,0.0,1,1


## 슬라이딩 윈도우

In [20]:
def extract_inputoutput_shift(df, lookback_time=2, predict_time=1):
    # 1) 과거 시점별 피처 생성
    feat_dfs = []
    for t in range(lookback_time):
        tmp = df.shift(t)
        feat_dfs.append(tmp)
    X = pd.concat(feat_dfs, axis=1)

    # 2) 예측 대상(타깃) 생성
    y = df[['congestion']].shift(-predict_time)

    # 3) NaN 제거 및 인덱스 리셋
    valid_idx = X.dropna().index.intersection(y.dropna().index)
    X = X.loc[valid_idx].reset_index(drop=True)
    y = y.loc[valid_idx].reset_index(drop=True)

    print("X, Y 데이터 분류 완료!")
    return X, y

In [21]:
x, t = extract_inputoutput_shift(train_data)
x

X, Y 데이터 분류 완료!


,station_number,direction,temp,wind_speed,rn_hr,humidity,solar,congestion,feel_temp,feel_stage,...,direction,temp,wind_speed,rn_hr,humidity,solar,congestion,feel_temp,feel_stage,is_holiday
0,150,0,-9.7,2.0,0.0,64.1,0.000000,0,-9.8,1,...,0.0,-9.6,3.3,0.0,64.1,0.000000,0.0,-12.6,0.0,1.0
1,150,0,-9.3,2.4,0.0,64.1,0.000000,1,-10.3,0,...,0.0,-9.7,2.0,0.0,64.1,0.000000,0.0,-9.8,1.0,1.0
2,150,0,-9.3,1.7,0.0,64.1,0.000000,2,-10.1,0,...,0.0,-9.3,2.4,0.0,64.1,0.000000,1.0,-10.3,0.0,1.0
3,150,0,-9.1,1.3,0.0,64.1,0.000000,3,-9.7,1,...,0.0,-9.3,1.7,0.0,64.1,0.000000,2.0,-10.1,0.0,1.0
4,150,0,-8.5,0.6,0.0,64.1,0.000000,3,-9.7,1,...,0.0,-9.1,1.3,0.0,64.1,0.000000,3.0,-9.7,1.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16369317,9006,1,2.6,0.3,0.0,86.6,0.030000,19,2.3,1,...,1.0,4.6,0.7,0.0,76.2,0.260000,18.0,4.1,1.0,1.0
16369318,9006,1,1.4,0.0,0.0,92.1,0.007959,15,1.3,1,...,1.0,2.6,0.3,0.0,86.6,0.030000,19.0,2.3,1.0,1.0
16369319,9006,1,0.8,0.5,0.0,95.3,0.000000,13,0.7,1,...,1.0,1.4,0.0,0.0,92.1,0.007959,15.0,1.3,1.0,1.0
16369320,9006,1,0.4,0.1,0.0,96.2,0.000000,14,0.2,1,...,1.0,0.8,0.5,0.0,95.3,0.000000,13.0,0.7,1.0,1.0


In [22]:
t

,congestion
0,1.0
1,2.0
2,3.0
3,3.0
4,5.0
...,...
16369317,15.0
16369318,13.0
16369319,14.0
16369320,14.0


In [23]:
x_train, x_test, t_train, t_test = train_test_split(x, t, test_size=0.3, shuffle=False)

print('x_train shape :', x_train.shape)
print('t_train shape :', t_train.shape)
print('x_test shape :', x_test.shape)
print('t_test shape :', t_test.shape)

x_train shape : (11458525, 22)
t_train shape : (11458525, 1)
x_test shape : (4910797, 22)
t_test shape : (4910797, 1)


In [24]:
timesteps = 2
feature = 11

x_train = np.array(x_train)
x_train = x_train.reshape(x_train.shape[0], timesteps, feature)

x_test = np.array(x_test)
x_test = x_test.reshape(x_test.shape[0], timesteps, feature)

t_train = np.array(t_train)
t_test = np.array(t_test)

print('reshape 후 x_train shape :', x_train.shape)
print('t_train shape :', t_train.shape)
print('reshape 후 x_test shape :', x_test.shape)
print('t_test shape :', t_test.shape)

reshape 후 x_train shape : (11458525, 2, 11)
t_train shape : (11458525, 1)
reshape 후 x_test shape : (4910797, 2, 11)
t_test shape : (4910797, 1)


In [25]:
from tensorflow.keras import regularizers
def gru_model(cell_size=128, learning_rate=0.001):
    timesteps = 2
    feature = 10
    model = Sequential(name='congestion_gru')
    model.add(GRU(cell_size, input_shape=(timesteps, feature), return_sequences=True,
                  kernel_regularizer=regularizers.l2(0.001)))
    model.add(GRU(cell_size, kernel_regularizer=regularizers.l2(0.001)))
    model.add(Dropout(0.2))
    model.add(Dense(1))
    model.compile(loss='mse', optimizer=Adam(learning_rate=learning_rate), metrics=[RootMeanSquaredError()])
    return model

In [26]:
temp_grue = gru_model()
temp_grue.summary()

/usr/local/lib/python3.11/dist-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "congestion_gru"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gru (GRU)                       │ (None, 2, 128)         │        53,760 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_1 (GRU)                     │ (None, 128)            │        99,072 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 152,961 (597.50 KB)

 Trainable params: 152,961 (597.50 KB)

 Non-trainable params: 0 (0.00 B)

In [27]:
import optuna
from scikeras.wrappers import KerasRegressor
from sklearn.model_selection import cross_val_score
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split
from tensorflow.keras import regularizers
#
early_stopping = EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)

#성능 평가 함수(scoring)로 평균제곱근오차 기본 제공이 아니라 커스텀
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))
rmse_scorer = make_scorer(rmse, greater_is_better=False)

#데이터가 너무 많아 10~20%만 추출하여 하이퍼파라미터 탐색하도록(더 빠르게)
x_train_subset, _, t_train_subset, _ = train_test_split(
    x_train, t_train, train_size=0.1, shuffle=False , random_state=42
)
def objective(trial):
    # Optuna에서 베이지안 방식으로 샘플링할 파라미터 정의
    cell_size = trial.suggest_categorical('cell_size', [128, 248])
    learning_rate = trial.suggest_float('learning_rate', 5e-4, 1e-3, log=True)
    epochs = trial.suggest_int('epochs', 15, 25)
    batch_size = trial.suggest_categorical('batch_size', [128, 256])

    # scikeras로 래핑
    Congestion_Gru_MODEL = KerasRegressor(
        model=gru_model,
        cell_size=cell_size,
        learning_rate=learning_rate,
        epochs=epochs,
        batch_size=batch_size,
        callbacks=[early_stopping]
    )
    # cross_val_score에서 cv=2와 데이터 직접 전달
    scores = cross_val_score(Congestion_Gru_MODEL, x_train_subset, t_train_subset, scoring=rmse_scorer, cv=2)
    return -scores.mean()  # minimize(RMSE)

In [28]:
bayesian_study = optuna.create_study(direction='minimize')

bayesian_study.optimize(objective, n_trials=20)

[I 2025-06-16 23:32:08,385] A new study created in memory with name: no-name-dbce7efa-10c2-45a6-a77e-99bd6904bb5c
/usr/local/lib/python3.11/dist-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/21
4476/4476 ━━━━━━━━━━━━━━━━━━━━ 26s 5ms/step - loss: 103.6231 - root_mean_squared_error: 10.0024
Epoch 2/21
  22/4476 ━━━━━━━━━━━━━━━━━━━━ 22s 5ms/step - loss: 74.0487 - root_mean_squared_error: 8.5638

/usr/local/lib/python3.11/dist-packages/keras/src/callbacks/early_stopping.py:153: UserWarning: Early stopping conditioned on metric `val_loss` which is not available. Available metrics are: loss,root_mean_squared_error
  current = self.get_monitor_value(logs)


4476/4476 ━━━━━━━━━━━━━━━━━━━━ 22s 5ms/step - loss: 66.5529 - root_mean_squared_error: 8.1155
Epoch 3/21
4476/4476 ━━━━━━━━━━━━━━━━━━━━ 22s 5ms/step - loss: 63.7367 - root_mean_squared_error: 7.9311
Epoch 4/21
4476/4476 ━━━━━━━━━━━━━━━━━━━━ 22s 5ms/step - loss: 60.1863 - root_mean_squared_error: 7.6925
Epoch 5/21
4476/4476 ━━━━━━━━━━━━━━━━━━━━ 22s 5ms/step - loss: 56.2682 - root_mean_squared_error: 7.4184
Epoch 6/21
4476/4476 ━━━━━━━━━━━━━━━━━━━━ 22s 5ms/step - loss: 54.6367 - root_mean_squared_error: 7.2918
Epoch 7/21
4476/4476 ━━━━━━━━━━━━━━━━━━━━ 22s 5ms/step - loss: 52.2961 - root_mean_squared_error: 7.1153
Epoch 8/21
4476/4476 ━━━━━━━━━━━━━━━━━━━━ 22s 5ms/step - loss: 50.5493 - root_mean_squared_error: 6.9784
Epoch 9/21
4476/4476 ━━━━━━━━━━━━━━━━━━━━ 22s 5ms/step - loss: 50.3934 - root_mean_squared_error: 6.9555
Epoch 10/21
4476/4476 ━━━━━━━━━━━━━━━━━━━━ 22s 5ms/step - loss: 49.0872 - root_mean_squared_error: 6.8509
Epoch 11/21
4476/4476 ━━━━━━━━━━━━━━━━━━━━ 22s 5ms/step - loss: 4

/usr/local/lib/python3.11/dist-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


...

/usr/local/lib/python3.11/dist-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


2238/2238 ━━━━━━━━━━━━━━━━━━━━ 13s 5ms/step - loss: 105.1843 - root_mean_squared_error: 10.0076
Epoch 2/16
  22/2238 ━━━━━━━━━━━━━━━━━━━━ 11s 5ms/step - loss: 60.8183 - root_mean_squared_error: 7.7451

/usr/local/lib/python3.11/dist-packages/keras/src/callbacks/early_stopping.py:153: UserWarning: Early stopping conditioned on metric `val_loss` which is not available. Available metrics are: loss,root_mean_squared_error
  current = self.get_monitor_value(logs)


2238/2238 ━━━━━━━━━━━━━━━━━━━━ 11s 5ms/step - loss: 53.9851 - root_mean_squared_error: 7.3121
Epoch 3/16
2238/2238 ━━━━━━━━━━━━━━━━━━━━ 11s 5ms/step - loss: 47.6223 - root_mean_squared_error: 6.8564
Epoch 4/16
2238/2238 ━━━━━━━━━━━━━━━━━━━━ 11s 5ms/step - loss: 43.6905 - root_mean_squared_error: 6.5557
Epoch 5/16
2238/2238 ━━━━━━━━━━━━━━━━━━━━ 11s 5ms/step - loss: 40.7723 - root_mean_squared_error: 6.3224
Epoch 6/16
2238/2238 ━━━━━━━━━━━━━━━━━━━━ 11s 5ms/step - loss: 38.9823 - root_mean_squared_error: 6.1725
Epoch 7/16
2238/2238 ━━━━━━━━━━━━━━━━━━━━ 11s 5ms/step - loss: 37.4218 - root_mean_squared_error: 6.0377
Epoch 8/16
2238/2238 ━━━━━━━━━━━━━━━━━━━━ 11s 5ms/step - loss: 36.6003 - root_mean_squared_error: 5.9626
Epoch 9/16
2238/2238 ━━━━━━━━━━━━━━━━━━━━ 11s 5ms/step - loss: 35.2438 - root_mean_squared_error: 5.8406
Epoch 10/16
2238/2238 ━━━━━━━━━━━━━━━━━━━━ 11s 5ms/step - loss: 34.7831 - root_mean_squared_error: 5.7940
Epoch 11/16
2238/2238 ━━━━━━━━━━━━━━━━━━━━ 11s 5ms/step - loss: 3

[I 2025-06-17 03:41:18,187] Trial 19 finished with value: 16.317944318911245 and parameters: {'cell_size': 248, 'learning_rate': 0.0005100726560266714, 'epochs': 16, 'batch_size': 256}. Best is trial 11 with value: 9.009451780255338.


In [29]:
print("Random Search에서의 최고의 하이퍼파라미터 조합 : ", bayesian_study.best_trial)

Random Search에서의 최고의 하이퍼파라미터 조합 :  FrozenTrial(number=11, state=1, values=[9.009451780255338], datetime_start=datetime.datetime(2025, 6, 17, 1, 47, 9, 618014), datetime_complete=datetime.datetime(2025, 6, 17, 2, 1, 21, 94460), params={'cell_size': 248, 'learning_rate': 0.000996069243535314, 'epochs': 18, 'batch_size': 128}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'cell_size': CategoricalDistribution(choices=(128, 248)), 'learning_rate': FloatDistribution(high=0.001, log=True, low=0.0005, step=None), 'epochs': IntDistribution(high=25, log=False, low=15, step=1), 'batch_size': CategoricalDistribution(choices=(128, 256))}, trial_id=11, value=None)


In [30]:
best_params = bayesian_study.best_params
final_gru_model = gru_model(
    cell_size=best_params['cell_size'],
    learning_rate=best_params['learning_rate']
)
history = final_gru_model.fit(
    x_train, t_train,
    epochs=best_params['epochs'],
    batch_size=best_params['batch_size'],
    callbacks=[early_stopping]
)

/usr/local/lib/python3.11/dist-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/18
89520/89520 ━━━━━━━━━━━━━━━━━━━━ 466s 5ms/step - loss: 111.5431 - root_mean_squared_error: 10.4858
Epoch 2/18


/usr/local/lib/python3.11/dist-packages/keras/src/callbacks/early_stopping.py:153: UserWarning: Early stopping conditioned on metric `val_loss` which is not available. Available metrics are: loss,root_mean_squared_error
  current = self.get_monitor_value(logs)


89520/89520 ━━━━━━━━━━━━━━━━━━━━ 463s 5ms/step - loss: 98.0022 - root_mean_squared_error: 9.7845
Epoch 3/18
89520/89520 ━━━━━━━━━━━━━━━━━━━━ 463s 5ms/step - loss: 96.7621 - root_mean_squared_error: 9.7140
Epoch 4/18
89520/89520 ━━━━━━━━━━━━━━━━━━━━ 463s 5ms/step - loss: 96.2472 - root_mean_squared_error: 9.6744
Epoch 5/18
89520/89520 ━━━━━━━━━━━━━━━━━━━━ 463s 5ms/step - loss: 92.9970 - root_mean_squared_error: 9.4994
Epoch 6/18
89520/89520 ━━━━━━━━━━━━━━━━━━━━ 463s 5ms/step - loss: 92.8894 - root_mean_squared_error: 9.4917
Epoch 7/18
89520/89520 ━━━━━━━━━━━━━━━━━━━━ 463s 5ms/step - loss: 93.2169 - root_mean_squared_error: 9.5105
Epoch 8/18
89520/89520 ━━━━━━━━━━━━━━━━━━━━ 464s 5ms/step - loss: 93.4938 - root_mean_squared_error: 9.5135
Epoch 9/18
89520/89520 ━━━━━━━━━━━━━━━━━━━━ 465s 5ms/step - loss: 92.1646 - root_mean_squared_error: 9.4400
Epoch 10/18
89520/89520 ━━━━━━━━━━━━━━━━━━━━ 465s 5ms/step - loss: 91.2180 - root_mean_squared_error: 9.3865
Epoch 11/18
89520/89520 ━━━━━━━━━━━━━━

In [31]:
loss, rmse = final_gru_model.evaluate(x_test, t_test, verbose=1)
print('test loss(MSE) :', round(loss, 6))
print('test RMSE :', round(rmse, 2))

153463/153463 ━━━━━━━━━━━━━━━━━━━━ 389s 3ms/step - loss: 103.6564 - root_mean_squared_error: 9.9351
test loss(MSE) : 121.74556
test RMSE : 10.87


In [32]:
final_gru_model.save("model_gru_last.keras")